# Chapter 10 — Bonus Exercise: Reward Shaping

**Session 4 | Chapter 10 | Bonus (~10 min, at home or after the Ch11 exercise)**

> **In RL, what you reward is what you get.** Reward design is one of the hardest parts of building RL systems.

We give you a *learning* agent (`train_q_learning`) as a black box — Chapter 11 explains how it works.
You change the **reward structure**, retrain, and look at the policy the agent learns.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

rng = np.random.default_rng(42)   # one seeded generator for everything → reproducible
print('Ready!')

## Setup — run these cells (GridWorld + a Q-learning agent as black box)

In [ ]:
# ── GridWorld: the SAME environment in Ch10, Ch11 and the animation notebook ──
# Cell types: 0 = Free, 1 = Hole, 2 = Goal, 3 = Start
# Layout is identical to Gymnasium's FrozenLake 4x4 map (SFFF / FHFH / FFFH / HFFG)
GRID = np.array([[3, 0, 0, 0],
                 [0, 1, 0, 1],
                 [0, 0, 0, 1],
                 [1, 0, 0, 2]])
ROWS, COLS = GRID.shape
N_STATES, N_ACTIONS = ROWS * COLS, 4                     # 16 states, 4 actions
ACTIONS = {0: 'Left', 1: 'Down', 2: 'Right', 3: 'Up'}   # same action coding as FrozenLake
MOVES   = {0: (0, -1), 1: (1, 0), 2: (0, 1), 3: (-1, 0)}  # (row delta, col delta)
REWARDS = {0: -0.01, 1: -1.0, 2: +1.0, 3: -0.01}         # free / hole / goal / start

def env_reset():
    """Start a new episode: the agent stands on the start cell (state 0)."""
    return 0

def env_step(state, action):
    """Apply an action. Returns (next_state, reward, done)."""
    r, c = divmod(state, COLS)
    dr, dc = MOVES[action]
    nr, nc = min(ROWS - 1, max(0, r + dr)), min(COLS - 1, max(0, c + dc))   # walls: stay put
    next_state = nr * COLS + nc
    cell = GRID[nr, nc]
    done = cell in (1, 2)                                                    # hole or goal ends the episode
    return next_state, REWARDS[cell], done

# ── Drawing helpers ──
CELL_COLORS = {0: '#eef3f5', 1: '#e74c3c', 2: '#2ecc71', 3: '#3498db'}
CELL_LABELS = {0: '', 1: 'H', 2: 'G', 3: 'S'}
ARROWS      = {0: '←', 1: '↓', 2: '→', 3: '↑'}

def draw_grid(ax, title=''):
    for r in range(ROWS):
        for c in range(COLS):
            ct = GRID[r, c]
            ax.add_patch(mpatches.FancyBboxPatch((c + 0.04, ROWS - 1 - r + 0.04), 0.92, 0.92,
                                                 boxstyle='round,pad=0.02', facecolor=CELL_COLORS[ct],
                                                 edgecolor='white', lw=2))
            if CELL_LABELS[ct]:
                ax.text(c + 0.5, ROWS - 1 - r + 0.5, CELL_LABELS[ct], ha='center', va='center',
                        fontsize=18, fontweight='bold', color='white')
            ax.text(c + 0.1, ROWS - 1 - r + 0.86, str(r * COLS + c), fontsize=8, color='gray', va='top')
    ax.set_xlim(0, COLS); ax.set_ylim(0, ROWS); ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title, fontsize=12)

def draw_policy(ax, Q, title='Greedy policy (arrows = argmax Q)'):
    """Arrows for the best action per state; '·' where nothing was learned yet."""
    draw_grid(ax, title)
    for s in range(N_STATES):
        r, c = divmod(s, COLS)
        if GRID[r, c] in (1, 2):
            continue
        learned = np.any(Q[s] != 0)
        ax.text(c + 0.5, ROWS - r - 0.5 - (0.2 if s == 0 else 0), ARROWS[int(np.argmax(Q[s]))] if learned else '·',
                ha='center', va='center', fontsize=22, fontweight='bold', color='#1a1a1a' if learned else 'gray')

def draw_trajectory(ax, traj, color='#1a1a1a'):
    xs = [s % COLS + 0.5 for s in traj]; ys = [ROWS - s // COLS - 0.5 for s in traj]
    ax.plot(xs, ys, '-', color=color, lw=2.5, alpha=0.6)
    ax.plot(xs[-1], ys[-1], '*', color='#f39c12', ms=18, markeredgecolor='#1a1a1a')

print(f'GridWorld ready: {N_STATES} states, {N_ACTIONS} actions {ACTIONS}')

In [ ]:
def run_episode(policy_fn, max_steps=100):
    """Play one episode with a policy (state -> action). Returns (trajectory, total_reward, reached_goal)."""
    state = env_reset()
    trajectory, total_reward = [state], 0.0
    for _ in range(max_steps):
        next_state, reward, done = env_step(state, policy_fn(state))
        trajectory.append(next_state)
        total_reward += reward
        state = next_state
        if done:
            break
    return trajectory, total_reward, (state == 15)

In [ ]:
def train_q_learning(n_episodes=3000, alpha=0.1, gamma=0.99, epsilon=1.0,
                     epsilon_decay=0.999, epsilon_min=0.01, max_steps=100, seed=42):
    """Tabular Q-learning on the GridWorld (explained in detail in Chapter 11).
    Returns the Q-table and a list with 1/0 per episode (goal reached or not)."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((N_STATES, N_ACTIONS))
    success_history = []
    for episode in range(n_episodes):
        state = env_reset()
        for _ in range(max_steps):
            if rng.random() < epsilon:                      # explore
                action = int(rng.integers(N_ACTIONS))
            else:                                           # exploit
                action = int(np.argmax(Q[state]))
            next_state, reward, done = env_step(state, action)
            target = reward + (0 if done else gamma * np.max(Q[next_state]))
            Q[state, action] += alpha * (target - Q[state, action])
            state = next_state
            if done:
                break
        success_history.append(float(done and reward > 0))
        epsilon = max(epsilon * epsilon_decay, epsilon_min)
    return Q, success_history

In [ ]:
def train_and_show(title):
    """Train with the CURRENT global REWARDS, print success rate and plot the learned policy."""
    Q, success = train_q_learning()
    greedy = lambda s: int(np.argmax(Q[s]))
    wins = np.mean([run_episode(greedy)[2] for _ in range(200)])
    print(f'{title}: rewards={REWARDS}')
    print(f'  greedy policy reaches the goal in {wins:.0%} of 200 test episodes')
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    draw_policy(ax, Q, f'{title}\ngoal reached: {wins:.0%}')
    plt.show()
    return Q, wins

DEFAULT_REWARDS = dict(REWARDS)
print('Helper ready.')

## Task 1 — Baseline (~2 min)
Train the agent with the default rewards and look at the learned policy.

In [ ]:
# TODO 1: call train_and_show('Default rewards')
# Then answer in a comment: which path does the agent take? Does it avoid the holes?

## Task 2 — A *positive* step reward (~3 min)

Someone thinks "let's motivate the agent: +0.05 for every step it survives".
Set the free-cell reward (keys `0` and `3`) to **+0.05**, keep hole −1 and goal +1, and retrain.

In [ ]:
# TODO 2a: REWARDS[0] = REWARDS[3] = +0.05 ; then train_and_show('Step reward +0.05')

# TODO 2b: In a comment: what does the agent do now, and why is this "reward hacking"?

## Task 3 — Expensive steps, cheap holes (~3 min)

Now the opposite: every step costs **−0.5**, but a hole only costs **−0.1** (goal still +1).

In [ ]:
# TODO 3a: reset REWARDS to DEFAULT_REWARDS, then set free/start = -0.5 and hole = -0.1 ; retrain

# TODO 3b: In a comment: where does the agent go, and why is that the "rational" thing to do here?

## Bonus — design your own
Find reward values under which the agent reaches the goal **and** takes the shortest path. Is there more than one solution?

In [ ]:
# TODO BONUS: your own reward design
REWARDS.clear(); REWARDS.update(DEFAULT_REWARDS)   # start from the defaults

---
**Solutions:** see `../04-solutions/ch10_rl_intro_solutions.ipynb`